# Extract Mood/Vibe Words From Titles

This notebook reads the Reddit CSV, extracts mood/vibe descriptor words from the `title` column, and writes a new CSV with a `title_descriptors` column placed right after `title`.

Examples:

- `movies or tv shows that feel like a warm summer` -> `warm, summer`
- `like a ride in the cool, foggy, lush forest` -> `cool, foggy, lush, forest`

In [ ]:
from pathlib import Path
import csv
import re
import sys

csv.field_size_limit(sys.maxsize)

DATA_FILENAME = 'reddit_posts_20260506_195100.csv'
DATA_PATH = next(
    path for path in [
        Path('dataset') / DATA_FILENAME,
        Path('../dataset') / DATA_FILENAME,
    ]
    if path.exists()
)

OUTPUT_PATH = DATA_PATH.with_name('reddit_posts_with_title_descriptors.csv')

DATA_PATH, OUTPUT_PATH

In [ ]:
# Words that are common in recommendation requests but are not mood/vibe descriptors.
STOPWORDS = {
    'a', 'an', 'and', 'any', 'anything', 'are', 'as', 'be', 'but', 'by', 'can',
    'do', 'does', 'dont', 'feel', 'feeling', 'feels', 'film', 'films', 'for',
    'from', 'get', 'give', 'gives', 'have', 'help', 'i', 'im', 'in', 'is', 'it',
    'kind', 'kinda', 'like', 'looking', 'me', 'movie', 'movies', 'need', 'of',
    'or', 'out', 'please', 'recommend', 'recommendations', 'ride', 'scene',
    'seen', 'series', 'show', 'shows', 'something', 'that', 'the', 'these',
    'this', 'to', 'tv', 'want', 'watch', 'whatever', 'where', 'with', 'you'
}

# Multi-word phrases to keep together when they appear in a title.
PHRASES = [
    'back alley', 'coming of age', 'cosmic horror', 'deep sea', 'fever dream',
    'folk horror', 'found footage', 'haunted house', 'liminal space',
    'magical realism', 'noir', 'psychological horror', 'road trip',
    'small town', 'slow burn', 'summer evening'
]

# Some useful singular/plural cleanup for setting words.
NORMALIZE_WORDS = {
    'forests': 'forest',
    'towns': 'town',
    'cities': 'city',
    'vibes': 'vibe',
}

def normalize_title(title):
    title = str(title or '').replace('\ufeff', '')
    title = title.lower()
    title = re.sub(r'[“”]', '"', title)
    title = re.sub(r'[’]', "'", title)
    title = re.sub(r"\bcan't\b", 'cannot', title)
    title = re.sub(r"\bwon't\b", 'will not', title)
    title = re.sub(r"n't\b", ' not', title)
    title = re.sub(r"[^a-z0-9'\s,/-]", ' ', title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title

def extract_title_descriptors(title):
    text = normalize_title(title)
    descriptors = []

    # Keep important multi-word descriptors first, then remove them to avoid duplicates.
    for phrase in PHRASES:
        if re.search(r'(?<!\w)' + re.escape(phrase) + r'(?!\w)', text):
            descriptors.append(phrase)
            text = re.sub(r'(?<!\w)' + re.escape(phrase) + r'(?!\w)', ' ', text)

    # Split hyphen/slash-separated descriptors, e.g. foggy/rainy or non-violent.
    text = text.replace('/', ' ').replace('-', ' ')
    tokens = re.findall(r"[a-z0-9']+", text)

    for token in tokens:
        token = token.strip("'")
        token = NORMALIZE_WORDS.get(token, token)
        if len(token) < 3:
            continue
        if token in STOPWORDS:
            continue
        if token not in descriptors:
            descriptors.append(token)

    return ', '.join(descriptors)

extract_title_descriptors('movies or tv shows that feel like a warm summer'), extract_title_descriptors('like a ride in the cool, foggy, lush forest')

In [ ]:
# Preview a few extracted rows before writing the full file.
with DATA_PATH.open(encoding='utf-8-sig', newline='') as csv_file:
    reader = csv.DictReader(csv_file)
    preview = []
    for row in reader:
        descriptors = extract_title_descriptors(row.get('title'))
        if descriptors:
            preview.append({
                'title': row.get('title', ''),
                'title_descriptors': descriptors,
            })
        if len(preview) == 20:
            break

preview

In [ ]:
with DATA_PATH.open(encoding='utf-8-sig', newline='') as input_file:
    reader = csv.DictReader(input_file)
    original_fieldnames = list(reader.fieldnames or [])

    output_fieldnames = []
    for fieldname in original_fieldnames:
        output_fieldnames.append(fieldname)
        if fieldname == 'title':
            output_fieldnames.append('title_descriptors')

    with OUTPUT_PATH.open('w', encoding='utf-8', newline='') as output_file:
        writer = csv.DictWriter(output_file, fieldnames=output_fieldnames)
        writer.writeheader()

        row_count = 0
        rows_with_descriptors = 0
        for row in reader:
            row['title_descriptors'] = extract_title_descriptors(row.get('title'))
            writer.writerow(row)
            row_count += 1
            rows_with_descriptors += bool(row['title_descriptors'])

row_count, rows_with_descriptors, OUTPUT_PATH